In [1]:
%pip install neo4j

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from neo4j import GraphDatabase

# Configuración de conexión
uri = "bolt://localhost:7687"
user = "neo4j"
password = "graf1234"

driver = GraphDatabase.driver(uri, auth=(user, password))

def ejecutar_carga_completa():
    with driver.session() as session:
        # 1. Limpieza total para asegurar la idempotencia del script
        session.run("MATCH (n) DETACH DELETE n")
        
        # 2. Creación de la red con nombres reales de Ciudad Guayana
        query = """
        // --- NODOS: Almacén (1) ---
        CREATE (a:Almacen {id: 'ALM_A', nombre: 'CD Logística Unare', x: 0, y: 10})

        // --- NODOS: Intersecciones (10) ---
        CREATE (i1:Interseccion {id: 'INT_1', nombre: 'Cruce Av. Bolívar', x: 4, y: 12})
        CREATE (i2:Interseccion {id: 'INT_2', nombre: 'Redoma Chilemex', x: 4, y: 8})
        CREATE (i3:Interseccion {id: 'INT_3', nombre: 'Semáforo Castillito', x: 7, y: 14})
        CREATE (i4:Interseccion {id: 'INT_4', nombre: 'Cruce Alta Vista', x: 7, y: 10})
        CREATE (i5:Interseccion {id: 'INT_5', nombre: 'Av. Atlántico Norte', x: 7, y: 6})
        CREATE (i6:Interseccion {id: 'INT_6', nombre: 'Distribuidor Los Olivos', x: 10, y: 15})
        CREATE (i7:Interseccion {id: 'INT_7', nombre: 'Redoma de la Piña', x: 10, y: 11})
        CREATE (i8:Interseccion {id: 'INT_8', nombre: 'Distribuidor Librepuer', x: 10, y: 7})
        CREATE (i9:Interseccion {id: 'INT_9', nombre: 'Av. Guayana Este', x: 12, y: 13})
        CREATE (i10:Interseccion {id: 'INT_10', nombre: 'Cruce San Félix', x: 12, y: 9})

        // --- NODOS: Puntos de Entrega (5) ---
        CREATE (e1:PuntoEntrega {id: 'ENT_1', nombre: 'Santo Tomé IV', x: 16, y: 14})
        CREATE (e2:PuntoEntrega {id: 'ENT_2', nombre: 'EPA Alta Vista', x: 16, y: 11})
        CREATE (e3:PuntoEntrega {id: 'ENT_3', nombre: 'Farmatodo Los Olivos', x: 16, y: 8})
        CREATE (e4:PuntoEntrega {id: 'ENT_4', nombre: 'Tienda Traki Guayana', x: 16, y: 5})
        CREATE (e5:PuntoEntrega {id: 'ENT_5', nombre: 'Orinokia Mall', x: 16, y: 2})

        // --- RELACIONES: Infraestructura Vial Unificada ---
        // Salidas desde el Almacén principal
        CREATE (a)-[:CONECTA_A {distancia: 5.0, tiempo_estimado: 10, estado_trafico: 0.1, capacidad_max_ton: 30.0}]->(i1)
        CREATE (a)-[:CONECTA_A {distancia: 5.2, tiempo_estimado: 12, estado_trafico: 0.4, capacidad_max_ton: 30.0}]->(i2)
        
        // Conexiones de la red interna de intersecciones
        CREATE (i1)-[:CONECTA_A {distancia: 3.5, tiempo_estimado: 7, estado_trafico: 0.2, capacidad_max_ton: 20.0}]->(i3)
        CREATE (i1)-[:CONECTA_A {distancia: 4.0, tiempo_estimado: 8, estado_trafico: 0.5, capacidad_max_ton: 25.0}]->(i4)
        CREATE (i2)-[:CONECTA_A {distancia: 4.5, tiempo_estimado: 9, estado_trafico: 0.1, capacidad_max_ton: 20.0}]->(i5)
        
        CREATE (i3)-[:CONECTA_A {distancia: 3.0, tiempo_estimado: 5, estado_trafico: 0.2, capacidad_max_ton: 15.0}]->(i6)
        CREATE (i4)-[:CONECTA_A {distancia: 3.0, tiempo_estimado: 5, estado_trafico: 0.2, capacidad_max_ton: 15.0}]->(i7)
        CREATE (i5)-[:CONECTA_A {distancia: 4.0, tiempo_estimado: 8, estado_trafico: 0.3, capacidad_max_ton: 20.0}]->(i8)
        
        CREATE (i6)-[:CONECTA_A {distancia: 2.5, tiempo_estimado: 4, estado_trafico: 0.1, capacidad_max_ton: 15.0}]->(i9)
        CREATE (i7)-[:CONECTA_A {distancia: 3.0, tiempo_estimado: 5, estado_trafico: 0.2, capacidad_max_ton: 15.0}]->(i10)
        
        // Rutas hacia los Puntos de Entrega Finales
        CREATE (i9)-[:CONECTA_A {distancia: 4.0, tiempo_estimado: 8, estado_trafico: 0.8, capacidad_max_ton: 12.0}]->(e1)
        CREATE (i7)-[:CONECTA_A {distancia: 5.5, tiempo_estimado: 10, estado_trafico: 0.2, capacidad_max_ton: 12.0}]->(e2)
        CREATE (i8)-[:CONECTA_A {distancia: 6.5, tiempo_estimado: 12, estado_trafico: 0.3, capacidad_max_ton: 20.0}]->(e3)
        CREATE (i10)-[:CONECTA_A {distancia: 4.0, tiempo_estimado: 6, estado_trafico: 0.1, capacidad_max_ton: 10.0}]->(e4)
        CREATE (i8)-[:CONECTA_A {distancia: 7.0, tiempo_estimado: 14, estado_trafico: 0.4, capacidad_max_ton: 25.0}]->(e5)
        """
        session.run(query)
        print("✅ Escenario logístico de 16 nodos (Ciudad Guayana) cargado con éxito.")

ejecutar_carga_completa()

✅ Escenario logístico de 16 nodos (Ciudad Guayana) cargado con éxito.


In [3]:
def proyectar_grafo():
    with driver.session() as session:
        # Borrar proyección si ya existe para evitar errores
        session.run("CALL gds.graph.drop('grafo_logistica', false)")
        
        # Crear la proyección con las propiedades necesarias para Dijkstra y A*
        query = """
        CALL gds.graph.project(
            'grafo_logistica',
            ['Almacen', 'Interseccion', 'PuntoEntrega'],
            {
            CONECTA_A: {
                orientation: 'NATURAL',
                properties: ['distancia', 'tiempo_estimado', 'estado_trafico', 'capacidad_max_ton']
            }
            },
            {
            nodeProperties: ['x', 'y'] // Necesario para la heurística de A*
            }
        )
        """
        result = session.run(query)
        print("✅ Grafo proyectado en memoria exitosamente.")

proyectar_grafo()

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)", position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'column': 1, 'offset': 0, 'line': 1}}> for query: "CALL gds.graph.drop('grafo_logistica', false)"


✅ Grafo proyectado en memoria exitosamente.


In [28]:
def forzar_proyeccion():
    with driver.session() as session:
        # Paso A: Eliminar proyecciones viejas
        session.run("CALL gds.graph.drop('grafo_logistica', false)")
        
        # Paso B: Proyección con mapeo detallado de propiedades (Requisito para GDS 2.6.9)
        query = """
        CALL gds.graph.project(
            'grafo_logistica',
            {
                Almacen: { label: 'Almacen', properties: ['x', 'y'] },
                Interseccion: { label: 'Interseccion', properties: ['x', 'y'] },
                PuntoEntrega: { label: 'PuntoEntrega', properties: ['x', 'y'] }
            },
            {
                CONECTA_A: {
                    type: 'CONECTA_A',
                    orientation: 'NATURAL',
                    properties: {
                        distancia: { property: 'distancia' },
                        tiempo_estimado: { property: 'tiempo_estimado' },
                        estado_trafico: { property: 'estado_trafico' }
                    }
                }
            }
        )
        """
        session.run(query)
        print("✅ Grafo proyectado con éxito usando mapeo explícito.")

forzar_proyeccion()

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)", position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'column': 1, 'offset': 0, 'line': 1}}> for query: "CALL gds.graph.drop('grafo_logistica', false)"


✅ Grafo proyectado con éxito usando mapeo explícito.


In [29]:
# Comparación entre ruta más corta (física) y la más inteligente (tráfico)
def ejecutar_comparativa():
    with driver.session() as session:
        # Se cambia el destino a ENT_2 (uno de los 5 nuevos puntos)
        query = """
        MATCH (startNode:Almacen {id: 'ALM_A'}), (endNode:PuntoEntrega {id: 'ENT_2'})
        
        // Algoritmo Dijkstra: Basado en distancia
        CALL gds.shortestPath.dijkstra.stream('grafo_logistica', {
            sourceNode: startNode,
            targetNode: endNode,
            relationshipWeightProperty: 'distancia'
        })
        YIELD totalCost AS km
        
        // Algoritmo A*: Basado en tiempo/tráfico
        CALL gds.shortestPath.astar.stream('grafo_logistica', {
            sourceNode: startNode,
            targetNode: endNode,
            latitudeProperty: 'y',
            longitudeProperty: 'x',
            relationshipWeightProperty: 'tiempo_estimado'
        })
        YIELD totalCost AS minutos
        
        RETURN km, minutos
        """
        result = session.run(query)
        res = result.single()
        if res:
            print(f"--- 📊 RESULTADOS DEL OPTIMIZADOR (Hacia Punto Entrega 2) ---")
            print(f"📏 Ruta Dijkstra (Física): {res['km']} km")
            print(f"⏱️ Ruta A* (Inteligente): {res['minutos']} min")

ejecutar_comparativa()

--- 📊 RESULTADOS DEL OPTIMIZADOR (Hacia Punto Entrega 2) ---
📏 Ruta Dijkstra (Física): 17.5 km
⏱️ Ruta A* (Inteligente): 33.0 min


In [22]:
#Consulta 1: Análisis de riesgo de tráfico
query1 = """
MATCH (a)-[r:CONECTA_A]->(b)
WITH r, a, b, r.estado_trafico AS trafico 
WHERE trafico > 0.7
RETURN a.nombre AS Origen, b.nombre AS Destino, trafico AS Nivel_Alerta
"""
with driver.session() as session:
    result = session.run(query1)
    print("--- 🚦 Tramos con Tráfico Crítico (> 0.7) ---")
    for record in result:
        print(f"Alerta: Tramo de {record['Origen']} a {record['Destino']} tiene Nivel {record['Nivel_Alerta']}")

--- 🚦 Tramos con Tráfico Crítico (> 0.7) ---
Alerta: Tramo de Av. Guayana Este a Santo Tomé IV tiene Nivel 0.8


In [23]:
# Consulta 2: Desglose de propiedades
query2 = """
UNWIND ['distancia', 'tiempo_estimado', 'estado_trafico', 'capacidad_max_ton'] AS propiedad
MATCH (n:Almacen)-[r:CONECTA_A]->()
RETURN propiedad, r[propiedad] AS Valor
"""
with driver.session() as session:
    result = session.run(query2)
    print("--- 📋 Desglose de Propiedades Dinámicas (Salidas de Almacén) ---")
    for record in result:
        print(f"Propiedad: {record['propiedad']} | Valor: {record['Valor']}")

--- 📋 Desglose de Propiedades Dinámicas (Salidas de Almacén) ---
Propiedad: distancia | Valor: 5.2
Propiedad: distancia | Valor: 5.0
Propiedad: tiempo_estimado | Valor: 12
Propiedad: tiempo_estimado | Valor: 10
Propiedad: estado_trafico | Valor: 0.4
Propiedad: estado_trafico | Valor: 0.1
Propiedad: capacidad_max_ton | Valor: 30.0
Propiedad: capacidad_max_ton | Valor: 30.0


In [24]:
# Consulta 3: Reporte de capacidad de carga
query3 = """
MATCH path = (a:Almacen)-[:CONECTA_A*1..5]->(p:PuntoEntrega)
WHERE ALL(r IN relationships(path) WHERE r.capacidad_max_ton >= 15)
RETURN [n in nodes(path) | n.nombre] as Ruta, length(path) as Saltos
"""
with driver.session() as session:
    result = session.run(query3)
    print("--- 🚚 Rutas Disponibles para Carga Pesada (>= 15 ton) ---")
    for record in result:
        print(f"Ruta: {' -> '.join(record['Ruta'])} | Saltos: {record['Saltos']}")

--- 🚚 Rutas Disponibles para Carga Pesada (>= 15 ton) ---
Ruta: CD Logística Unare -> Redoma Chilemex -> Av. Atlántico Norte -> Distribuidor Librepuer -> Farmatodo Los Olivos | Saltos: 4
Ruta: CD Logística Unare -> Redoma Chilemex -> Av. Atlántico Norte -> Distribuidor Librepuer -> Orinokia Mall | Saltos: 4


In [25]:
# Consulta 4: Cálculo de costo de ruta personalizado
query4 = """
MATCH (a)-[r:CONECTA_A]->(b)
WITH a, b, (r.distancia * (1 + r.estado_trafico)) AS costo_calculado
RETURN a.nombre AS Origen, b.nombre AS Destino, costo_calculado
"""
with driver.session() as session:
    result = session.run(query4)
    print("--- 💰 Análisis de Costos por Tramo (Distancia x Tráfico) ---")
    for record in result:
        print(f"De {record['Origen']} a {record['Destino']}: ${record['costo_calculado']:.2f}")

--- 💰 Análisis de Costos por Tramo (Distancia x Tráfico) ---
De CD Logística Unare a Cruce Av. Bolívar: $5.50
De CD Logística Unare a Redoma Chilemex: $7.28
De Cruce Av. Bolívar a Semáforo Castillito: $4.20
De Cruce Av. Bolívar a Cruce Alta Vista: $6.00
De Redoma Chilemex a Av. Atlántico Norte: $4.95
De Semáforo Castillito a Distribuidor Los Olivos: $3.60
De Cruce Alta Vista a Redoma de la Piña: $3.60
De Av. Atlántico Norte a Distribuidor Librepuer: $5.20
De Distribuidor Los Olivos a Av. Guayana Este: $2.75
De Redoma de la Piña a Cruce San Félix: $3.60
De Av. Guayana Este a Santo Tomé IV: $7.20
De Redoma de la Piña a EPA Alta Vista: $6.60
De Distribuidor Librepuer a Farmatodo Los Olivos: $8.45
De Cruce San Félix a Tienda Traki Guayana: $4.40
De Distribuidor Librepuer a Orinokia Mall: $9.80


In [26]:
# Consulta 5: Auditoría de la proyección
query5 = """
CALL gds.graph.list() 
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""
with driver.session() as session:
    result = session.run(query5)
    print("--- 🛠️ Estado de la Proyección GDS en Memoria ---")
    for record in result:
        print(f"Grafo: {record['graphName']} | Nodos: {record['nodeCount']} | Relaciones: {record['relationshipCount']}")

--- 🛠️ Estado de la Proyección GDS en Memoria ---
Grafo: grafo_logistica | Nodos: 16 | Relaciones: 15
